In [0]:
-- ==============================================================================
-- FASE 3: MODELAGEM DE NEGÓCIO E AGREGAÇÕES (CAMADA GOLD COM SPARK SQL)
-- ==============================================================================

-- INDICADOR 1: FATURAMENTO MENSAL HISTÓRICO
-- O objetivo é saber quanto o e-commerce vende por mês/ano.
CREATE OR REPLACE TABLE workspace.default.gold_faturamento_mensal AS
SELECT 
    DATE_TRUNC('month', o.order_purchase_timestamp) AS mes_ano,
    COUNT(DISTINCT o.order_id) AS total_pedidos,
    ROUND(SUM(i.price), 2) AS receita_produtos,
    ROUND(SUM(i.freight_value), 2) AS gasto_frete,
    ROUND(SUM(i.price + i.freight_value), 2) AS faturamento_total
FROM workspace.default.silver_orders o
INNER JOIN workspace.default.silver_order_items i 
    ON o.order_id = i.order_id
WHERE o.order_status = 'delivered' -- Considera apenas pedidos entregues com sucesso
GROUP BY 1
ORDER BY mes_ano DESC;


-- INDICADOR 2: PERFORMANCE LOGÍSTICA POR ESTADO
-- O objetivo é descobrir o tempo médio real de entrega (em dias) para cada estado.
CREATE OR REPLACE TABLE workspace.default.gold_performance_logistica AS
SELECT 
    c.customer_state AS estado_cliente,
    COUNT(DISTINCT o.order_id) AS total_entregas,
    -- Calcula a diferença em dias entre a compra e a entrega real
    ROUND(AVG(DATEDIFF(o.order_delivered_customer_date, o.order_purchase_timestamp)), 1) AS media_dias_entrega,
    -- Calcula a diferença entre a entrega estimada e a real (valores positivos = atraso)
    ROUND(AVG(DATEDIFF(o.order_delivered_customer_date, o.order_estimated_delivery_date)), 1) AS media_desvio_estimativa
FROM workspace.default.silver_orders o
INNER JOIN workspace.default.silver_customers c 
    ON o.customer_id = c.customer_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY c.customer_state
ORDER BY media_dias_entrega DESC;


-- INDICADOR 3: RANKING DOS TOP 10 CLIENTES FIEIS (WINDOW FUNCTION)
-- Usando DENSE_RANK para classificar os clientes que mais gastaram no e-commerce.
CREATE OR REPLACE TABLE workspace.default.gold_ranking_clientes AS
WITH cte_gasto_clientes AS (
    SELECT 
        c.customer_unique_id AS id_unico_cliente,
        c.customer_state AS estado_cliente,
        ROUND(SUM(i.price), 2) AS total_gasto_produtos,
        COUNT(DISTINCT o.order_id) AS quantidade_pedidos
    FROM workspace.default.silver_orders o
    INNER JOIN workspace.default.silver_order_items i ON o.order_id = i.order_id
    INNER JOIN workspace.default.silver_customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id, c.customer_state
),
cte_ranking AS (
    SELECT 
        id_unico_cliente,
        estado_cliente,
        total_gasto_produtos,
        quantidade_pedidos,
        DENSE_RANK() OVER (ORDER BY total_gasto_produtos DESC) AS posicao_ranking
    FROM cte_gasto_clientes
)
SELECT * FROM cte_ranking 
WHERE posicao_ranking <= 10;